# KGGen — Ingesta por batches — COSORA

Notebook **autocontenido** para generar grafos con `kg-gen` (fuente `original`).

**Batch 1** ya existe en Drive (`graph_actas_e5_original_sub10_bal_raw.json`) — no regenerar salvo `FORCE_REGEN=True`.

**Batch 2+:** procesa actas 11–20, 21–30, … y guarda JSON + manifest.

**Prerequisito:** actas `.doc`/`.docx` en Drive (`RAG_UPC_Final_project`) o `data/raw` local.

Tras generar un batch, cargarlo en Neo4j con `neo4j_graph_rag.ipynb` (carga incremental).


## 0. Setup


In [ ]:
%pip install -q kg-gen python-dotenv python-docx pandas

import shutil
import subprocess
if shutil.which("antiword") is None:
    try:
        subprocess.run(["apt-get", "install", "-y", "-q", "antiword"], check=False)
    except Exception:
        print("⚠️  antiword no instalado — solo .docx")

import json
import os
import re
import sys
import time
from collections import defaultdict
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
RUNTIME = "colab" if IN_COLAB else "local"

# ─── Config batch ─────────────────────────────────────────────────────
KG_SOURCE = "original"
BATCH_SIZE = 10
BATCH_ID = 1              # batch 1 ya hecho; cambiar a 3, 4…
FORCE_REGEN = False       # True para regenerar aunque exista JSON

KG_MODEL = "openai/gpt-4o-mini"
KG_CONTEXT = (
    "Actas de obra ferroviaria en España. Personas, empresas (UTE, DF, DEO), "
    "elementos constructivos, incidencias, fechas."
)
KG_CHUNK_SIZE = 5000
KG_CLUSTER = False        # original: False (rápido); True si hay mucho ruido
BALANCED_SAMPLING = False # True solo si el batch mezcla familias desiguales

RAW_DOCS_PATH_OVERRIDE = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

from dotenv import load_dotenv
if RUNTIME == "colab":
    load_dotenv("/content/drive/MyDrive/variablentorno/.env")
elif Path(".env").exists():
    load_dotenv(".env")
elif Path("../../.env").exists():
    load_dotenv("../../.env")

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY no encontrada")


def resolve_paths(runtime):
    if runtime == "colab":
        docs_dir = "/content/drive/MyDrive/RAG_UPC_Final_project"
        graph_dir = f"{docs_dir}/graph"
    else:
        nb_dir = Path(".").resolve()
        root = nb_dir.parents[1] if nb_dir.name == "experiments" else (
            nb_dir.parent if nb_dir.name == "notebooks" else nb_dir
        )
        docs_dir = str(root / "data" / "raw")
        graph_dir = str(root / "data" / "graph")
    batch_dir = str(Path(graph_dir) / "batches")
    Path(graph_dir).mkdir(parents=True, exist_ok=True)
    Path(batch_dir).mkdir(parents=True, exist_ok=True)
    return docs_dir, graph_dir, batch_dir


DOCS_DIR, GRAPH_DIR, BATCH_DIR = resolve_paths(RUNTIME)
print(f"RUNTIME={RUNTIME}")
print(f"DOCS_DIR={DOCS_DIR}")
print(f"GRAPH_DIR={GRAPH_DIR}")
print(f"BATCH_ID={BATCH_ID}  BATCH_SIZE={BATCH_SIZE}")


## 1. Selección de actas del batch


In [ ]:
def list_acta_files(raw_dir: str) -> list[Path]:
    raw = Path(raw_dir)
    files = sorted(list(raw.glob("*.docx")) + list(raw.glob("*.doc")))
    if not files:
        raise FileNotFoundError(f"Sin .doc/.docx en {raw_dir}")
    return files


def batch_slice(files: list[Path], batch_id: int, batch_size: int) -> list[Path]:
    start = (batch_id - 1) * batch_size
    end = start + batch_size
    return files[start:end]


def batch_json_path(batch_id: int) -> Path:
    return Path(GRAPH_DIR) / f"graph_actas_e5_original_batch{batch_id:02d}_bal_raw.json"


def batch_manifest_path(batch_id: int) -> Path:
    return Path(BATCH_DIR) / f"batch_{batch_id:02d}_manifest.json"


all_files = list_acta_files(
    RAW_DOCS_PATH_OVERRIDE or DOCS_DIR
)
batch_files = batch_slice(all_files, BATCH_ID, BATCH_SIZE)
if not batch_files:
    raise ValueError(
        f"Batch {BATCH_ID} vacío (total actas={len(all_files)}). "
        f"Índice start={(BATCH_ID-1)*BATCH_SIZE}"
    )

manifest = {
    "batch_id": BATCH_ID,
    "batch_size": BATCH_SIZE,
    "sources": [f.stem for f in batch_files],
    "files": [str(f) for f in batch_files],
}
with open(batch_manifest_path(BATCH_ID), "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(f"Total actas en corpus: {len(all_files)}")
print(f"Batch {BATCH_ID}: actas {len(batch_files)}")
for f in batch_files:
    print(f"  • {f.name}")
print(f"Manifest: {batch_manifest_path(BATCH_ID)}")


## 2. Extracción con KGGen


In [ ]:
from docx import Document
import subprocess as sp
from kg_gen import KGGen


def _normalize(t):
    return re.sub(r"\s+", " ", t).strip()


def _extract_docx(fp: Path) -> str | None:
    try:
        doc = Document(fp)
    except Exception as e:
        print(f"  ⚠️ {fp.name}: {e}")
        return None
    parts = []
    for p in doc.paragraphs:
        t = _normalize(p.text)
        if t:
            parts.append(t)
    for table in doc.tables:
        for row in table.rows:
            row_texts, seen = [], set()
            for cell in row.cells:
                t = _normalize(cell.text)
                if t and t not in seen:
                    seen.add(t)
                    row_texts.append(t)
            if row_texts:
                parts.append(" || ".join(row_texts))
    return "\n".join(parts).strip() or None


def _extract_doc(fp: Path) -> str | None:
    try:
        r = sp.run(["antiword", str(fp)], capture_output=True, text=True)
        return r.stdout.strip() if r.returncode == 0 else None
    except Exception as e:
        print(f"  ⚠️ antiword falló {fp.name}: {e}")
        return None


def extract_texts(files: list[Path]) -> tuple[list[str], list[str]]:
    texts, sources = [], []
    for fp in files:
        raw = _extract_docx(fp) if fp.suffix.lower() == ".docx" else _extract_doc(fp)
        if raw:
            texts.append(raw)
            sources.append(fp.stem)
    return texts, sources


def graph_to_dict(graph, sources, texts):
    return {
        "entities": sorted(list(graph.entities)),
        "edges": sorted(list(graph.edges)),
        "relations": [list(r) for r in graph.relations],
        "entity_clusters": {k: sorted(list(v)) for k, v in (graph.entity_clusters or {}).items()},
        "edge_clusters": {k: sorted(list(v)) for k, v in (graph.edge_clusters or {}).items()},
        "meta": {
            "model": KG_MODEL,
            "kg_source": KG_SOURCE,
            "batch_id": BATCH_ID,
            "batch_size": BATCH_SIZE,
            "n_inputs": len(texts),
            "sources": sources,
            "context": KG_CONTEXT,
            "cluster": KG_CLUSTER,
        },
    }


out_path = batch_json_path(BATCH_ID)
if out_path.exists() and not FORCE_REGEN:
    print(f"📁 Ya existe {out_path.name} — no se regenera (FORCE_REGEN=False)")
    with open(out_path, encoding="utf-8") as f:
        gd = json.load(f)
else:
    if BATCH_ID == 1:
        print("ℹ️  Batch 1 ya generado como sub10 en graph_rag.ipynb — usa ese JSON o FORCE_REGEN=True")
    texts, sources = extract_texts(batch_files)
    print(f"Extrayendo {len(texts)} actas con kg-gen...")
    kg = KGGen(model=KG_MODEL, temperature=0.0, api_key=api_key)
    t0 = time.perf_counter()
    partial_graphs = []
    for i, txt in enumerate(texts, 1):
        ti = time.perf_counter()
        g_i = kg.generate(input_data=txt, context=KG_CONTEXT, chunk_size=KG_CHUNK_SIZE, cluster=False)
        partial_graphs.append(g_i)
        print(
            f"  {i}/{len(texts)} {sources[i-1][:40]:<40} "
            f"+{len(g_i.relations):>4} triples ({time.perf_counter()-ti:.1f}s)"
        )
    graph = kg.aggregate(partial_graphs)
    if KG_CLUSTER and graph.relations:
        print("Clustering global...")
        graph = kg.cluster(graph, context=KG_CONTEXT)
    gd = graph_to_dict(graph, sources, texts)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(gd, f, ensure_ascii=False, indent=2)
    print(f"✅ {len(gd['relations'])} triples en {(time.perf_counter()-t0)/60:.1f} min → {out_path}")

print(f"Triples: {len(gd['relations'])}")
print(f"Actas:   {len(gd['meta']['sources'])}")


## 3. Siguiente paso

1. Subir/copiar el JSON a `GRAPH_DIR` si generaste en local
2. En `neo4j_graph_rag.ipynb`: apuntar `GRAPH_JSON_NAME` al nuevo batch y usar carga incremental (`WIPE_BEFORE_LOAD=False`)
3. Repetir con `BATCH_ID = 3`, …
